# 01 — External Database Setup (CultPass)

Builds the **external** SQLite database that simulates the data UDA-Hub
receives from its first customer, **CultPass**. This is the data the
agentic system queries via the `cultpass_*` tools to enrich tickets.

Schema:

- `CultPassMember` — members and tier (classic / plus / elite)
- `CultPassEvent` — events the member can book
- `CultPassBooking` — bookings + plus-ones
- `CultPassPayment` — subscription / event / refund payments

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data.external import db, seed
from agentic.config import settings
settings.external_db_path

## 1. (Re)create the database and load seed rows

In [ ]:
counts = seed.seed_all(reset=True)
counts

## 2. Inspect schema

In [ ]:
from tabulate import tabulate
tables = db.fetch_all(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
)
print(tabulate(tables, headers='keys'))

## 3. Sample rows

In [ ]:
print(tabulate(db.fetch_all('SELECT member_id, full_name, email, tier, status, city FROM CultPassMember'), headers='keys'))

In [ ]:
print(tabulate(db.fetch_all('SELECT event_id, title, city, starts_at, capacity, category FROM CultPassEvent'), headers='keys'))

In [ ]:
print(tabulate(db.fetch_all('''
    SELECT b.booking_id, b.member_id, e.title AS event, b.status, b.plus_one
      FROM CultPassBooking b JOIN CultPassEvent e USING(event_id)
      ORDER BY b.booked_at DESC
'''), headers='keys'))

In [ ]:
print(tabulate(db.fetch_all('SELECT payment_id, member_id, amount_cents, kind, status FROM CultPassPayment'), headers='keys'))

## 4. Spot-check: Bob's duplicate charge

Bob (`cp_m_002`) has two captured subscription payments in May — the seed
scenario the Resolver/refund tool will fix in notebook 03.

In [ ]:
print(tabulate(db.fetch_all('''
    SELECT * FROM CultPassPayment WHERE member_id="cp_m_002" ORDER BY created_at DESC
'''), headers='keys'))

---

External database is ready. Continue to `02_core_db_setup.ipynb`.